# Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor

/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Model

In [2]:
class Config:
    seed = 42
    test_size = 0.2
    batch_size = 64
    lr = 0.001
    weight_decay = 1e-4
    dropout_rate = 0.3
    patience = 15
    epochs = 200
    
torch.manual_seed(Config.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
def load_data():
    df = pd.read_csv('encoded_scaled_df.csv', index_col=0)
    df = df.drop(['Unnamed: 0.1', 'Unnamed: 0', 'Sales'], axis=1, errors='ignore')
    
    X = df.drop('Price', axis=1).values
    y = df['Price'].values.reshape(-1, 1)
    
    return X, y

X, y = load_data()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=Config.test_size, random_state=Config.seed
)

def to_tensor(data, device):
    return torch.FloatTensor(data).to(device)

X_train_tensor = to_tensor(X_train, device)
y_train_tensor = to_tensor(y_train, device)
X_test_tensor = to_tensor(X_test, device)
y_test_tensor = to_tensor(y_test, device)

class EnhancedPricePredictor(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(0.1),
            nn.Dropout(0.15),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x)

input_size = X_train.shape[1]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EnhancedPricePredictor(input_size).to(device)
model.load_state_dict(torch.load('price_predictor.pth'))
model.eval()

def evaluate_model():
    model.eval()
    with torch.no_grad():
        test_preds = model(X_test_tensor).cpu().numpy()
        test_true = y_test_tensor.cpu().numpy()
        
        r2 = r2_score(test_true, test_preds)
        
    print(f'\nFinal Metric')
    print(f'R²: {r2:.4f}')

evaluate_model()


Final Metric
R²: 0.8971


/var/folders/pf/dxhtqvm567j60pyv1yzqr_sc0000gn/T/ipykernel_28063/682153786.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('price_predi

# Hybrid model

In [4]:
model.eval()
with torch.no_grad():
    train_deep_preds = model(X_train_tensor).cpu().numpy().ravel()
    test_deep_preds = model(X_test_tensor).cpu().numpy().ravel()

X_train_hybrid = np.hstack([X_train, train_deep_preds.reshape(-1, 1)])
X_test_hybrid = np.hstack([X_test, test_deep_preds.reshape(-1, 1)])

pd.DataFrame(X_train_hybrid)

,0,1,2,3,4,5,6,7,8,9,...,1298,1299,1300,1301,1302,1303,1304,1305,1306,1307
0,3.0,0.691823,-0.673429,0.092399,-0.068804,1.0,-0.789936,-0.065502,-0.619098,0.137540,...,11.0,14.0,2.0,4.0,319.0,46.0,0.0,0.0,0.0,-0.390082
1,2.0,0.247177,-0.201993,0.081362,-0.134145,3.0,1.086817,1.064598,1.368981,1.255997,...,1.0,5.0,6.0,1.0,5.0,1.0,0.0,0.0,0.0,-0.062554
2,0.0,1.581116,-0.673429,-0.127220,-0.521199,1.0,-0.789936,-0.217069,-0.282483,0.137540,...,5.0,5.0,3.0,2.0,125.0,18.0,0.0,0.0,0.0,-0.644036
3,3.0,-0.197470,2.626627,-0.233209,-0.546384,3.0,-0.789936,-0.217069,-0.282483,-0.144382,...,3.0,9.0,4.0,1.0,69.0,10.0,0.0,0.0,0.0,-0.604192
4,3.0,2.025763,0.269444,-0.116700,-0.445403,1.0,-0.789936,-0.217069,-0.282483,0.137540,...,9.0,20.0,5.0,3.0,263.0,38.0,0.0,0.0,0.0,-0.601544
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40643,0.0,-0.642117,2.626627,-2.136910,0.098468,0.0,-0.789936,-1.486722,-1.478288,-1.560354,...,5.0,13.0,0.0,2.0,133.0,20.0,0.0,0.0,0.0,-0.244587
40644,3.0,-1.086763,-0.673429,-0.186675,-0.521436,3.0,-0.789936,-0.782425,-1.478288,-1.560354,...,9.0,16.0,6.0,3.0,260.0,37.0,0.0,0.0,0.0,-0.466798
40645,3.0,2.470410,-0.673429,-0.137426,-0.488885,3.0,-0.789936,1.683741,1.368981,2.114747,...,6.0,7.0,4.0,2.0,158.0,23.0,0.0,0.0,0.0,-0.639086
40646,2.0,2.025763,-0.664001,0.514903,3.727126,0.0,1.414984,-0.356121,-0.619098,-0.987320,...,9.0,8.0,0.0,3.0,251.0,37.0,0.0,0.0,0.0,0.249768


In [5]:
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5],
    'iterations': [500, 1000, 1500]
}

catboost_model = CatBoostRegressor(
    loss_function='RMSE',
    verbose=10,
    early_stopping_rounds=20,
    random_seed=Config.seed,
    task_type="GPU" if torch.cuda.is_available() else "CPU"
)

grid_search_result = catboost_model.grid_search(
    param_grid,
    X=X_train_hybrid,
    y=y_train.ravel(),
    cv=3,
    shuffle=True,
    stratified=False,
    partition_random_seed=Config.seed,
    plot=False
)

print("Best parameters found:")
print(catboost_model.get_params())

0:	learn: 0.9961865	test: 0.9613227	best: 0.9613227 (0)	total: 65.3ms	remaining: 32.6s
10:	learn: 0.9127714	test: 0.8810539	best: 0.8810539 (10)	total: 94.3ms	remaining: 4.19s
20:	learn: 0.8381757	test: 0.8094364	best: 0.8094364 (20)	total: 125ms	remaining: 2.85s
30:	learn: 0.7705591	test: 0.7446371	best: 0.7446371 (30)	total: 154ms	remaining: 2.33s
40:	learn: 0.7096669	test: 0.6862199	best: 0.6862199 (40)	total: 187ms	remaining: 2.09s
50:	learn: 0.6541401	test: 0.6331401	best: 0.6331401 (50)	total: 217ms	remaining: 1.91s
60:	learn: 0.6043004	test: 0.5854844	best: 0.5854844 (60)	total: 249ms	remaining: 1.79s
70:	learn: 0.5599531	test: 0.5430550	best: 0.5430550 (70)	total: 281ms	remaining: 1.7s
80:	learn: 0.5205357	test: 0.5052434	best: 0.5052434 (80)	total: 312ms	remaining: 1.61s
90:	learn: 0.4849407	test: 0.4712387	best: 0.4712387 (90)	total: 341ms	remaining: 1.53s
100:	learn: 0.4532379	test: 0.4409058	best: 0.4409058 (100)	total: 371ms	remaining: 1.47s
110:	learn: 0.4245213	test: 0.4

400:	learn: 0.2061618	test: 0.2155360	best: 0.2155360 (400)	total: 1.44s	remaining: 355ms
410:	learn: 0.2057932	test: 0.2154427	best: 0.2154384 (409)	total: 1.47s	remaining: 318ms
420:	learn: 0.2054392	test: 0.2153364	best: 0.2153364 (420)	total: 1.5s	remaining: 281ms
430:	learn: 0.2050646	test: 0.2151610	best: 0.2151382 (427)	total: 1.53s	remaining: 245ms
440:	learn: 0.2046860	test: 0.2148521	best: 0.2148521 (440)	total: 1.56s	remaining: 208ms
450:	learn: 0.2042895	test: 0.2146998	best: 0.2146996 (449)	total: 1.59s	remaining: 173ms
460:	learn: 0.2038553	test: 0.2146037	best: 0.2145841 (459)	total: 1.62s	remaining: 137ms
470:	learn: 0.2035438	test: 0.2146194	best: 0.2145672 (466)	total: 1.65s	remaining: 102ms
480:	learn: 0.2032313	test: 0.2146035	best: 0.2145672 (466)	total: 1.68s	remaining: 66.4ms
490:	learn: 0.2028394	test: 0.2143572	best: 0.2143572 (490)	total: 1.71s	remaining: 31.4ms
499:	learn: 0.2025381	test: 0.2142781	best: 0.2142781 (499)	total: 1.74s	remaining: 0us

bestTest =

100:	learn: 0.2265599	test: 0.2245878	best: 0.2245878 (100)	total: 380ms	remaining: 1.5s
110:	learn: 0.2250513	test: 0.2235947	best: 0.2235947 (110)	total: 416ms	remaining: 1.46s
120:	learn: 0.2239015	test: 0.2228775	best: 0.2228775 (120)	total: 450ms	remaining: 1.41s
130:	learn: 0.2229097	test: 0.2224613	best: 0.2224613 (130)	total: 480ms	remaining: 1.35s
140:	learn: 0.2219192	test: 0.2218556	best: 0.2218459 (139)	total: 515ms	remaining: 1.31s
150:	learn: 0.2209908	test: 0.2212474	best: 0.2212474 (150)	total: 549ms	remaining: 1.27s
160:	learn: 0.2202971	test: 0.2209287	best: 0.2209287 (160)	total: 588ms	remaining: 1.24s
170:	learn: 0.2196065	test: 0.2202757	best: 0.2202757 (170)	total: 631ms	remaining: 1.21s
180:	learn: 0.2190759	test: 0.2199161	best: 0.2199161 (180)	total: 663ms	remaining: 1.17s
190:	learn: 0.2185651	test: 0.2196631	best: 0.2196631 (190)	total: 697ms	remaining: 1.13s
200:	learn: 0.2180071	test: 0.2194807	best: 0.2194806 (199)	total: 729ms	remaining: 1.08s
210:	learn:

160:	learn: 0.3255863	test: 0.3188355	best: 0.3188355 (160)	total: 547ms	remaining: 1.15s
170:	learn: 0.3122486	test: 0.3061243	best: 0.3061243 (170)	total: 589ms	remaining: 1.13s
180:	learn: 0.3009176	test: 0.2953623	best: 0.2953623 (180)	total: 623ms	remaining: 1.1s
190:	learn: 0.2908171	test: 0.2857552	best: 0.2857552 (190)	total: 654ms	remaining: 1.06s
200:	learn: 0.2822711	test: 0.2776587	best: 0.2776587 (200)	total: 686ms	remaining: 1.02s
210:	learn: 0.2749424	test: 0.2707090	best: 0.2707090 (210)	total: 717ms	remaining: 983ms
220:	learn: 0.2685467	test: 0.2645911	best: 0.2645911 (220)	total: 747ms	remaining: 943ms
230:	learn: 0.2630881	test: 0.2593710	best: 0.2593710 (230)	total: 784ms	remaining: 912ms
240:	learn: 0.2584782	test: 0.2549989	best: 0.2549989 (240)	total: 820ms	remaining: 881ms
250:	learn: 0.2544585	test: 0.2511634	best: 0.2511634 (250)	total: 854ms	remaining: 848ms
260:	learn: 0.2510119	test: 0.2478244	best: 0.2478244 (260)	total: 889ms	remaining: 814ms
270:	learn:

80:	learn: 0.2220065	test: 0.2220008	best: 0.2220008 (80)	total: 253ms	remaining: 1.31s
90:	learn: 0.2205331	test: 0.2212135	best: 0.2211861 (89)	total: 284ms	remaining: 1.28s
100:	learn: 0.2196732	test: 0.2204452	best: 0.2204452 (100)	total: 315ms	remaining: 1.24s
110:	learn: 0.2182349	test: 0.2196826	best: 0.2196826 (110)	total: 357ms	remaining: 1.25s
120:	learn: 0.2173060	test: 0.2192895	best: 0.2192895 (120)	total: 387ms	remaining: 1.21s
130:	learn: 0.2164463	test: 0.2192187	best: 0.2192187 (130)	total: 417ms	remaining: 1.17s
140:	learn: 0.2153123	test: 0.2184827	best: 0.2184813 (139)	total: 447ms	remaining: 1.14s
150:	learn: 0.2143501	test: 0.2183692	best: 0.2183692 (150)	total: 477ms	remaining: 1.1s
160:	learn: 0.2136717	test: 0.2179796	best: 0.2179796 (160)	total: 507ms	remaining: 1.07s
170:	learn: 0.2129222	test: 0.2175924	best: 0.2175808 (168)	total: 536ms	remaining: 1.03s
180:	learn: 0.2121635	test: 0.2173110	best: 0.2173110 (180)	total: 565ms	remaining: 997ms
190:	learn: 0.2

...

680:	learn: 0.1605800	test: 0.2056649	best: 0.2056386 (678)	total: 7.4s	remaining: 8.9s
690:	learn: 0.1602507	test: 0.2057019	best: 0.2056386 (678)	total: 7.5s	remaining: 8.79s
700:	learn: 0.1597558	test: 0.2054950	best: 0.2054950 (700)	total: 7.68s	remaining: 8.76s
710:	learn: 0.1592056	test: 0.2054786	best: 0.2054729 (706)	total: 7.8s	remaining: 8.65s
720:	learn: 0.1585358	test: 0.2053439	best: 0.2053439 (720)	total: 7.9s	remaining: 8.54s
730:	learn: 0.1581808	test: 0.2052944	best: 0.2052838 (728)	total: 8s	remaining: 8.42s
740:	learn: 0.1578720	test: 0.2053012	best: 0.2052838 (728)	total: 8.11s	remaining: 8.3s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 0.2052837864
bestIteration = 728

76:	loss: 0.2052838	best: 0.2043872 (56)	total: 6m 7s	remaining: 19.1s
0:	learn: 0.9173980	test: 0.8858565	best: 0.8858565 (0)	total: 13.3ms	remaining: 19.9s
10:	learn: 0.4090754	test: 0.3999771	best: 0.3999771 (10)	total: 144ms	remaining: 19.4s
20:	learn: 0.2642779	test: 0.2635

450:	learn: 0.2181466	test: 0.2241262	best: 0.2241262 (450)	total: 4.87s	remaining: 11.3s
460:	learn: 0.2174269	test: 0.2237045	best: 0.2237045 (460)	total: 4.98s	remaining: 11.2s
470:	learn: 0.2167736	test: 0.2234010	best: 0.2234010 (470)	total: 5.08s	remaining: 11.1s
480:	learn: 0.2161813	test: 0.2231370	best: 0.2231370 (480)	total: 5.18s	remaining: 11s
490:	learn: 0.2156225	test: 0.2229716	best: 0.2229716 (490)	total: 5.28s	remaining: 10.9s
500:	learn: 0.2150481	test: 0.2226519	best: 0.2226473 (499)	total: 5.39s	remaining: 10.7s
510:	learn: 0.2144790	test: 0.2223723	best: 0.2223723 (510)	total: 5.49s	remaining: 10.6s
520:	learn: 0.2139663	test: 0.2220642	best: 0.2220642 (520)	total: 5.6s	remaining: 10.5s
530:	learn: 0.2134414	test: 0.2217802	best: 0.2217802 (530)	total: 5.7s	remaining: 10.4s
540:	learn: 0.2129444	test: 0.2215240	best: 0.2215240 (540)	total: 5.83s	remaining: 10.3s
550:	learn: 0.2125143	test: 0.2213533	best: 0.2213533 (550)	total: 5.92s	remaining: 10.2s
560:	learn: 0.

1360:	learn: 0.1901661	test: 0.2121245	best: 0.2121240 (1357)	total: 14.9s	remaining: 1.52s
1370:	learn: 0.1900059	test: 0.2120447	best: 0.2120445 (1369)	total: 15s	remaining: 1.41s
1380:	learn: 0.1898507	test: 0.2120182	best: 0.2120176 (1379)	total: 15.1s	remaining: 1.3s
1390:	learn: 0.1897288	test: 0.2119708	best: 0.2119664 (1386)	total: 15.2s	remaining: 1.19s
1400:	learn: 0.1896019	test: 0.2119426	best: 0.2119393 (1398)	total: 15.3s	remaining: 1.08s
1410:	learn: 0.1894365	test: 0.2118786	best: 0.2118786 (1410)	total: 15.4s	remaining: 972ms
1420:	learn: 0.1893511	test: 0.2118669	best: 0.2118669 (1420)	total: 15.5s	remaining: 863ms
1430:	learn: 0.1892573	test: 0.2118546	best: 0.2118526 (1427)	total: 15.6s	remaining: 753ms
1440:	learn: 0.1891001	test: 0.2118275	best: 0.2118236 (1439)	total: 15.7s	remaining: 644ms
1450:	learn: 0.1889351	test: 0.2117848	best: 0.2117846 (1449)	total: 15.8s	remaining: 534ms
1460:	learn: 0.1888093	test: 0.2117474	best: 0.2117472 (1459)	total: 15.9s	remainin

760:	learn: 0.1651919	test: 0.2060435	best: 0.2060326 (751)	total: 7.82s	remaining: 7.59s
770:	learn: 0.1649547	test: 0.2061356	best: 0.2060326 (751)	total: 7.92s	remaining: 7.48s
780:	learn: 0.1643085	test: 0.2057828	best: 0.2057828 (780)	total: 8.02s	remaining: 7.38s
790:	learn: 0.1641499	test: 0.2057908	best: 0.2057820 (781)	total: 8.12s	remaining: 7.27s
800:	learn: 0.1636873	test: 0.2057154	best: 0.2057154 (800)	total: 8.21s	remaining: 7.17s
810:	learn: 0.1633585	test: 0.2056574	best: 0.2056522 (808)	total: 8.31s	remaining: 7.06s
820:	learn: 0.1629723	test: 0.2055636	best: 0.2055588 (818)	total: 8.42s	remaining: 6.96s
830:	learn: 0.1625797	test: 0.2054755	best: 0.2054755 (830)	total: 8.51s	remaining: 6.85s
840:	learn: 0.1622214	test: 0.2054971	best: 0.2054566 (831)	total: 8.61s	remaining: 6.75s
850:	learn: 0.1619401	test: 0.2054451	best: 0.2054189 (847)	total: 8.71s	remaining: 6.64s
860:	learn: 0.1613503	test: 0.2053257	best: 0.2053178 (858)	total: 8.81s	remaining: 6.54s
870:	learn

40:	learn: 0.2180909	test: 0.2288395	best: 0.2288395 (40)	total: 419ms	remaining: 4.69s
50:	learn: 0.2114217	test: 0.2257842	best: 0.2257842 (50)	total: 522ms	remaining: 4.59s
60:	learn: 0.2062901	test: 0.2239398	best: 0.2239398 (60)	total: 622ms	remaining: 4.48s
70:	learn: 0.2018765	test: 0.2221623	best: 0.2221623 (70)	total: 725ms	remaining: 4.38s
80:	learn: 0.1984745	test: 0.2210416	best: 0.2210270 (79)	total: 825ms	remaining: 4.27s
90:	learn: 0.1952103	test: 0.2201999	best: 0.2201999 (90)	total: 930ms	remaining: 4.18s
100:	learn: 0.1928543	test: 0.2193557	best: 0.2193557 (100)	total: 1.03s	remaining: 4.06s
110:	learn: 0.1894484	test: 0.2186801	best: 0.2186801 (110)	total: 1.13s	remaining: 3.95s
120:	learn: 0.1866876	test: 0.2180825	best: 0.2180825 (120)	total: 1.22s	remaining: 3.83s
130:	learn: 0.1838186	test: 0.2171935	best: 0.2171639 (128)	total: 1.33s	remaining: 3.75s
140:	learn: 0.1809678	test: 0.2165343	best: 0.2165343 (140)	total: 1.42s	remaining: 3.61s
150:	learn: 0.1791125	

In [6]:
catboost_model.fit(
    X_train_hybrid, y_train.ravel(),
    eval_set=(X_test_hybrid, y_test.ravel())
)

hybrid_preds = catboost_model.predict(X_test_hybrid)
hybrid_r2 = r2_score(y_test, hybrid_preds)

print(f'Hybrid Model R² Score (after hyperparameter tuning): {hybrid_r2:.4f}')

0:	learn: 0.9088848	test: 0.9207887	best: 0.9207887 (0)	total: 8.88ms	remaining: 4.43s
10:	learn: 0.4012811	test: 0.4469788	best: 0.4469788 (10)	total: 93.8ms	remaining: 4.17s
20:	learn: 0.2594015	test: 0.3323849	best: 0.3323849 (20)	total: 180ms	remaining: 4.1s
30:	learn: 0.2264619	test: 0.3112058	best: 0.3112058 (30)	total: 272ms	remaining: 4.11s
40:	learn: 0.2163318	test: 0.3048322	best: 0.3048322 (40)	total: 356ms	remaining: 3.99s
50:	learn: 0.2098332	test: 0.3009317	best: 0.3009317 (50)	total: 439ms	remaining: 3.87s
60:	learn: 0.2053592	test: 0.2979670	best: 0.2979670 (60)	total: 522ms	remaining: 3.76s
70:	learn: 0.2018275	test: 0.2957869	best: 0.2957869 (70)	total: 607ms	remaining: 3.67s
80:	learn: 0.1981237	test: 0.2934764	best: 0.2934764 (80)	total: 687ms	remaining: 3.56s
90:	learn: 0.1947054	test: 0.2910824	best: 0.2910824 (90)	total: 771ms	remaining: 3.46s
100:	learn: 0.1918697	test: 0.2897233	best: 0.2897233 (100)	total: 852ms	remaining: 3.37s
110:	learn: 0.1893972	test: 0.2